# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library. All schema entities (record sets, fields, columns) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Title: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description (via attribute access, not subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
List all available record sets (tables) and their fields using their `@id` as references.

Below, we display the available record set `@id`s, each field's `@id`, and their user-friendly `name`. This information helps you select the correct entities for extraction and further processing.

In [ ]:
# List all record sets in the dataset
for record_set in dataset.metadata.record_sets:
    print(f"RecordSet @id: {record_set['@id']}, name: {record_set.get('name', 'No name')}")
    if 'fields' in record_set and isinstance(record_set['fields'], list):
        for field in record_set['fields']:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', 'No name')}")
    print('-' * 60)

## 3. Data Extraction
We'll extract data from each record set into pandas DataFrames using their `@id`. All entity references use their Croissant-assigned `@id`, for full reproducibility.

In [ ]:
# Collect all record set @ids
record_sets = [r['@id'] for r in dataset.metadata.record_sets]
dataframes = {}

# Load each record set as a DataFrame
for record_set_id in record_sets:
    print(f"Extracting records from record set: {record_set_id}")
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records.")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))
    print('-' * 80)

if not dataframes:
    print("No dataframes loaded. Please check the schema or the data availability.")

## 4. Exploratory Data Analysis (EDA)
Let's select one record set with substantive numeric fields, filter entries, normalize, and group for basic analysis.

_Note: All columns and grouping are referenced by `@id` (from the Croissant schema overview section above)._

We will use the first available record set and try to select the first numeric field found for demonstration. Adjust the variables as appropriate for your analysis and data context.

In [ ]:
import numpy as np

# Select the first record set for the demo
record_set_id = record_sets[0] if record_sets else None
df = dataframes.get(record_set_id)
if df is None or df.empty:
    print('No data available for EDA.')
else:
    # Try to find a numeric field (int or float columns)
    numeric_field_id = None
    for col in df.columns:
        # Try to infer numeric columns
        try:
            if np.issubdtype(df[col].dropna().dtype, np.number):
                numeric_field_id = col
                break
        except Exception:
            continue
    if not numeric_field_id:
        print('No numeric field found for EDA.')
    else:
        threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as demo threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group field that is categorical (object or string column)
        group_field_id = None
        for col in df.columns:
            if col == numeric_field_id:
                continue
            # Simple heuristic for likely categorical
            if df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No string or categorical field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and the effect of normalization. If a group field is found, we'll make a simple group barplot.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is None or df.empty or not numeric_field_id:
    print('No visualization possible: missing numeric field or data.')
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True, bins=30)
        plt.title(f"Normalized {numeric_field_id} (filtered)")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel('Frequency')
        plt.show()

    # Group visualization if available
    if 'group_field_id' in locals() and group_field_id is not None:
        grp = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grp, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (filtered)")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, analyze, and visualize a Croissant-structured FAIR^2 dataset using the `mlcroissant` library. We referenced all entities by their Croissant `@id` for proper provenance.

- The dataset provides mass spectrometry data for protein analysis in human mast cell-derived extracellular vesicles.
- Users can select and analyze all available fields and record sets, and the Croissant metadata structure aids automation and reproducibility.

We encourage further domain-specific exploration (e.g., statistical tests, biological interpretations) relevant to your application.